In [1]:
import openai
from typing import List

client = openai.OpenAI() # OpenAI API에 요청을 보내고 응답받는 기능을 제공하는 객체
response = client.chat.completions.create( # OpenAI의 Chat API를 사용하여 대화를 생성
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕하세요!"}]
)
response.choices[0].message.content

'안녕하세요! 어떻게 도와드릴까요?'

In [2]:
prompt_template = "주제 {topic}에 대해 짧은 설명을 해주세요."
def call_chat_model(messages: List[dict]):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
    )
    return response.choices[0].message.content

def invoke_chain(topic: str):
    prompt_value = prompt_template.format(topic=topic)
    messages = [{'role':'user','content':prompt_value}]
    return call_chat_model(messages)
invoke_chain("더블딥")

'더블딥(Double Dip)은 경제학에서 주로 사용되는 용어로, 경기 침체가 일어난 후 잠시 회복세를 보이다가 다시 한 번 심각한 침체에 빠지는 상황을 의미합니다. 이 패턴은 GDP 성장률이 두 번 연속으로 마이너스를 기록하는 현상으로 나타나며, 보통 경제가 완전히 회복되지 못하고 불황이 지속되는 경우에 발생합니다. 더블딥은 기업과 소비자 신뢰에 부정적인 영향을 미치며, 경제 정책 결정자들에게도 큰 도전 과제가 됩니다.'

In [3]:
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

load_dotenv(".env")
api_key = os.getenv("OPENAI_API_KEY")
llm = OpenAI(api_key=api_key)
llm.invoke("안녕")

'하세요! 저는 김민지입니다. 만나서 반가워요. 저는 대학교에서 국어국문학을 전공하고 있어요. 책 읽는 것과 영화 보는 것을 좋아해서 여가 시간에는 주로 그 둘을 즐기고 있어요. 또한 요리하는 것도 취미로 즐기고 있어요. 앞으로 많은 이야기를 나눌 수 있으면 좋겠어요. 잘 부탁드려요!'

In [5]:
from langchain_openai import ChatOpenAI #랭체인에서 OpenAI의 GPT 모델을 사용하는 클래스
from langchain_core.prompts import ChatPromptTemplate # 프롬프트 템플릿을 만들어주는 클래스
from langchain_core.output_parsers import StrOutputParser # GPT모델이 반환한 응답을 문자열로 반환, 모델의 응답은 다양한 형태로 반환될 수 있는데, 그 중에서 문자열 응답만 추출하여 반환
from langchain_core.runnables import RunnablePassthrough # 입력 데이터를 그대로 통과시키는 역할
from dotenv import load_dotenv

prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧은 설명을 해주세요.")
output_parser = StrOutputParser() # 출력 파서를 문자열로 설정
model = ChatOpenAI(model='gpt-4o')
# 파이프라인 설정: 주제를 받아 프롬프트를 생성하고, 모델로 응답을 생성한 후 문자열로 파싱
chain = (
    {"topic":RunnablePassthrough()} # 입력받은 주제를 그대로 통과
    | prompt # 프롬프트 템플릿 적용
    | model #모델을 사용하여 응답 생성
    | StrOutputParser() # 응답을 문자열로 파싱
)
chain.invoke("더블딥")

'"더블딥(Double Dip)"은 경제학에서 주로 사용되는 용어로, 경기 침체가 끝나고 잠시 회복되는 듯하다가 다시 침체로 접어드는 상황을 나타냅니다. 이는 경제가 V자형 회복을 보이지 않고, W자형 패턴을 띠며 두 번의 침체를 겪는 것을 설명하는 데 사용됩니다. 더블딥은 소비자 신뢰 저하, 투자 감소, 정부 정책 실패 등 다양한 요인에 의해 발생할 수 있으며, 경제에 장기적인 부정적 영향을 미칠 수 있습니다.'

In [6]:
model = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧은 설명을 해주세요.")
parser = StrOutputParser()
chain = prompt | model | parser # 프롬프트, 모델, 출력 파서를 체인으로 연결
chain.invoke({"topic":"더블딥"}) # 단일 주제
chain.batch([{"topic":"더블딥"}, {"topic":"인플레이션"}]) # 여러 주제를 동시에 처리
for token in chain.stream({"topic":"더블딥"}): #응답을 토큰 단위로 스트리밍하여 출력
    print(token, end="", flush=True) # 스트리밍된 내용을 출력, 각 내용을 붙여서 출력하며 버퍼를 즉시 플러시하여 실시간으로 보여줌

더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 두 번 발생하는 현상을 의미합니다. 첫 번째 침체가 발생한 후 경기가 잠시 회복되는 듯하다가 다시 심각한 침체로 빠지는 경우를 지칭합니다. 이러한 패턴은 소비자 신뢰도와 경제 활동의 불확실성으로 인해 발생할 수 있으며, 투자와 고용에도 부정적인 영향을 미칠 수 있습니다. 더블딥은 경제 정책 결정자들에게 어려운 도전 과제가 될 수 있습니다.

In [7]:
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해줘  : {answer}")
# 순서 : "answer"의 키에 chain에서 생성된 응답이 들어가고, 이 응답을 영어로 번역하는 프롬프트로 전달, 이 후 모델을 실행하여 번역된 응답을 생성하고, 결과를 StrOutputParser를 통해 문자열로 변환
composed_chain = {"answer":chain} | analysis_prompt | model | parser
composed_chain.invoke({"topic":"더블딥"})

'The term "Double Dip" in economics refers to a phenomenon where a recession occurs, followed by a brief recovery, only for the economy to deteriorate again. This pattern is generally contrasted with V-shaped recovery and U-shaped recovery. A Double Dip can typically arise from factors such as a decline in consumer confidence, economic uncertainty, and external shocks, indicating a situation that requires economic policy responses.'

In [8]:
# 함수를 러너블로 강제 변환
model = ChatOpenAI(model='gpt-4o-mini')
prompt = ChatPromptTemplate.from_template("주제 {topic}에 대해 짧은 설명을 해주세요.")
parser = StrOutputParser()
chain = prompt | model | parser
analysis_prompt = ChatPromptTemplate.from_template("이 대답을 영어로 번역해줘  : {answer}")

composed_chain_with_lambda=(
    chain # 정의한 chain을 사용하여 입력된 데이터에 대한 설명을 받음
    | (lambda input: {"answer":input}) #설명을 "answer"의 key로 변환
    | analysis_prompt # 'answer'의 key를 영어로 번역하도록 하는 프롬프트에 전달
    | model # 모델이 번역한 결과 생성
    | StrOutputParser() # 문자열로 파싱
)

composed_chain_with_lambda.invoke({"topic":"더블딥"})

'The term "double dip" describes a phenomenon that occurs in the economy or financial markets, primarily referring to situations where a recession happens twice. Initially, the economy enters a recession and then shows signs of recovery for a certain period, but this recovery is not sustained, and the economy declines again. This second downturn can occur due to incomplete economic recovery or external factors. Double dips can lead to negative effects on the economy, such as reduced consumer confidence and investment.'

In [9]:
# __or__을 사용한 파이썬 오버로딩
class CustomLCEL:
    def __init__(self, value):
        self.value = value

    def __or__(self,other):
        if callable(other): # other가 함수일 경우, 함수를 호출하고, 그 결과를 새로운 객체로 반환
            return CustomLCEL(other(self.value))
        else: #other가 함수가 아니면 오류
            raise ValueError("Right operand must be callable")
    def result(self):
        return self.value # 현재 값 반환

def add_exclamation(s): #문자열 끝 느낌표 추가
    return s + "!"
def reverse_string(s): #문자열 뒤집기
    return s[::-1]

custom_chain=(
    CustomLCEL("랭체인 공부하기")
    | add_exclamation
    | reverse_string
)

result = custom_chain.result()
print(result)

!기하부공 인체랭


In [10]:
# 파이프 메소드 방법 1
composed_chain_with_pipe =(
    chain
    .pipe(lambda input: {"answer":input})
    .pipe(analysis_prompt)
    .pipe(model)
    .pipe(StrOutputParser())
)

composed_chain_with_pipe.invoke({'topic':"더블딥"})


#방법 2
composed_chain_with_pipe = chain.pipe(lambda input:{"answer":input}, analysis_prompt, model, StrOutputParser())
composed_chain_with_pipe.invoke({"topic":"더블딥"})


'"Double Dip" is a term used in economics to describe a situation where the economy appears to have recovered, only to enter another recession. It generally refers to instances where there are repeated cycles of decline and recovery, rather than following a V-shaped recovery followed by a U-shaped or L-shaped recovery. A double dip can occur when the economic recovery is temporary and the economy continues to face difficulties due to structural problems or external shocks. This phenomenon can be significantly influenced by government economic policies or market responses.'

In [12]:
# RunnableParallel을 통한 체인 구성
from langchain_core.runnables import RunnableParallel

model = ChatOpenAI()
kor_chain=(
    ChatPromptTemplate.from_template("{topic}에 대해서 짧게 설명해줘.")
    | model
    | StrOutputParser()
)

eng_chain=(
    ChatPromptTemplate.from_template("{topic}에 대해 짧게 영어로 설명해줘.")
    | model
    | StrOutputParser()
)

# 병렬 실행을 위한 RunnableParallel 설정
parallel_chain = RunnableParallel(kor=kor_chain, eng=eng_chain)
result = parallel_chain.invoke({"topic":"더블딥"})

print("한글 설명: ",result['kor'])
print("영어 설명: ",result['eng'])

한글 설명:  더블딥은 대략적인 정의로 데뷔곡과 최신곡, 노래 가사 및 뮤직비디오를 이용해 가사의 사운드 및 레퍼런스를 분석한 결과를 매력적으로 제시하는 음악평론 브랜드이다. 실제로 진행자들이 직접 가사를 불러주는 등 어떤 노래에 대해 설명하고 음악에 관한 다양한 정보들을 함께 제공하는 프로그램으로 인기를 끌고 있다.
영어 설명:  Double deep learning (DoubleDip) is a technique that combines two deep learning models to improve performance and accuracy. By leveraging the strengths of two different models, DoubleDip can achieve better results in tasks such as image recognition, natural language processing, and more.


In [13]:
# 문자열 프롬프트 템플릿
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("주제 {topic}에 대해서 금융과 관련된 간략한 조언을 해줘.")
prompt_template.invoke({"topic":"투자"})

StringPromptValue(text='주제 투자에 대해서 금융과 관련된 간략한 조언을 해줘.')

In [15]:
# 챗 프롬프트 템플릿
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([ #ChatPromptTemplate: 사용자와 시스템 간의 메시지 포함
    ("system", "당신은 유능한 금융 조언가입니다."), # 시스템 메시지: AI의 역할을 정의, AI가 어떤 종류의 응답을 제공해야 하는지 정의
    ("user", "주제 {topic}에 대해 금융 관련 조언을 해주세요.") # 사용자 메시지: 사용자가 요청하는 내용을 포함하여, AI에게 특정 정보를 요청
])
prompt_template.invoke({"topic":"주식"})

ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='주제 주식에 대해 금융 관련 조언을 해주세요.', additional_kwargs={}, response_metadata={})])

In [17]:
# 매사자 자리 표시자
from langchain_core.prompts import ChatMessagePromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

# 방법 1: 메시지 자리 표시자를 포함한 ChatPromptTemplate 설정
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 유능한 금융 조언가입니다."),
    MessagesPlaceholder("msgs")
])
prompt_template.invoke({"msgs":[HumanMessage(content="안녕하세요!")]}) # 메시지 리스트를 'msgs' 자리 표시자에 전달하여 호출


ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={})])

In [18]:
# 방법 2: MessagePlaceholder 클래스를 사용하지 않고 비슷한 작업 수행
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 유능한 금융 조언가입니다."),
    ("placeholder","{msgs}") # msgs가 자리 표시자로 사용
])

prompt_template.invoke({"msgs":[HumanMessage(content='안녕하세요!')]})

ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={})])

In [19]:
# 퓨삿 예제 프롬프트 템플릿
example_prompt = PromptTemplate.from_template("질문: {question}\n답변: {answer}")
examples = [{
    "question":"주식 투자와 예금 중 어느 것이 더 수익률이 높은가?",
    "answer" : """
    후속 질문이 필요하신가요: 네.
    후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
    중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
    후속 질문: 예금의 평균 이자율은 얼마인가요?
    중간 답변: 예금의 평균 이자율은 연 1%입니다.
    따라서 최종 답변은: 주식투자
    """,
},
{
    "question":"부동산과 채권 중 어느 것이 더 안정적인 투자처인가?",
    "answer": """
    후속 질문이 필요한가요: 네.
    후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
    중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
    후속 질문: 채권의 위험도는 어느 정도인가요?
    중간 답변: 채권의 위험도는 낮은 편입니다.
    따라서 최종답변은: 채권
    """,
},
]

print(example_prompt.invoke(examples[0]).to_string())

질문: 주식 투자와 예금 중 어느 것이 더 수익률이 높은가?
답변: 
    후속 질문이 필요하신가요: 네.
    후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
    중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
    후속 질문: 예금의 평균 이자율은 얼마인가요?
    중간 답변: 예금의 평균 이자율은 연 1%입니다.
    따라서 최종 답변은: 주식투자
    


In [22]:
# 퓨샷 프롬프트 템플릿
from langchain_core.prompts import FewShotPromptTemplate
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="질문: {input}",
    input_variables=['input'],
)

print(prompt.invoke({"input": "부동산 투자의 장점은 무엇인가?"}).to_string()
)

질문: 주식 투자와 예금 중 어느 것이 더 수익률이 높은가?
답변: 
    후속 질문이 필요하신가요: 네.
    후속 질문: 주식 투자의 평균 수익률은 얼마인가요?
    중간 답변: 주식 투자의 평균 수익률은 연 7%입니다.
    후속 질문: 예금의 평균 이자율은 얼마인가요?
    중간 답변: 예금의 평균 이자율은 연 1%입니다.
    따라서 최종 답변은: 주식투자
    

질문: 부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
답변: 
    후속 질문이 필요한가요: 네.
    후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
    중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
    후속 질문: 채권의 위험도는 어느 정도인가요?
    중간 답변: 채권의 위험도는 낮은 편입니다.
    따라서 최종답변은: 채권
    

질문: 부동산 투자의 장점은 무엇인가?


In [28]:
# 예제 선택기 
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings

example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples, # 사용할 예제 목록
    OpenAIEmbeddings(api_key=api_key), # 임베딩 생성에 사용하는 클래스
    Chroma, # 임베딩을 저장하고 유사도 검색을 수행하는 벡터 저장소 클래스
    k=1, # 선택할 예제 갯수
)
question = "부동산 투자의 장점은 무엇인가?"
selected_examples = example_selector.select_examples({"question":question})
print(f"입력 질문:{question}")
for example in selected_examples:
    print("\n")
    print("# 입력과 가장 유사한 예제:")
    for k, v in example.items():
        print(f"{k}:{v}")

입력 질문:부동산 투자의 장점은 무엇인가?


# 입력과 가장 유사한 예제:
question:부동산과 채권 중 어느 것이 더 안정적인 투자처인가?
answer:
    후속 질문이 필요한가요: 네.
    후속 질문: 부동산 투자의 위험도는 어느 정도인가요?
    중간 답변: 부동산 투자의 위험도는 중간 수준입니다.
    후속 질문: 채권의 위험도는 어느 정도인가요?
    중간 답변: 채권의 위험도는 낮은 편입니다.
    따라서 최종답변은: 채권
    


In [ ]:
# 퓨샷 프롬프트 + 실제 모델과 함께 사용하는 예시 코드
example_prompt = PromptTemplate(
    input_variables=["question",'answer'], # 해당 템플릿은 2개의 입력 변수를 사용
    template='질문: {question}\n답변: {answer}' # 실제로 질문과 답변을 표시하는 형식
)

prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt = example_prompt, # 앞서 정의한 질문과 답변 형식의 프롬프트 템플릿을 사용하여 예제를 제공할 수 있도록 함
    prefix="다음은 금융 관련 질문과 답변의 예입니다:", # 프롬프트 앞부분에 붙는 텍스트
    suffix="질문: {input}\n답변:", # 질문 후 모델이 답변을 생성해야 할 부분
    input_variables=['input'] # 실제로 사용자가 입력한 질문
)

model = ChatOpenAI(model_name='gpt-4o')
chain = prompt | model
response = chain.invoke({"input":"부동산 투자의 장점은 무엇인가?"})
print(response.content)

부동산 투자의 장점은 여러 가지가 있습니다. 다음은 그 중 몇 가지 주요 장점을 설명합니다:

1. **자산의 가치 상승**: 부동산은 시간에 따라 가치가 상승할 가능성이 높습니다. 특히 인구 증가 지역이나 개발 가능성이 높은 곳에서는 자산 가치가 크게 상승할 수 있습니다.

2. **안정적 수익**: 임대 수익을 통한 정기적인 수입이 가능합니다. 이는 특히 은퇴 후 고정 수입원이 될 수 있습니다.

3. **물가 상승에 대한 헤지**: 부동산은 인플레이션에 대한 자연스러운 헤지 수단이 될 수 있습니다. 물가가 상승하면 대체로 부동산 가치와 임대료가 함께 상승하는 경향이 있습니다.

4. **자산의 물리적 안정성**: 주식이나 채권과 달리 부동산은 물리적인 자산으로, 실질적인 소유물이므로 파산 위험이 상대적으로 낮습니다.

5. **세금 혜택**: 많은 국가에서 부동산 투자에 대해 다양한 세금 혜택을 제공합니다. 예를 들어, 모기지 이자나 일정한 유지보수 비용을 공제 받을 수 있습니다.

부동산 투자는 이러한 장점을 통해 다양한 방식으로 투자자에게 이익을 제공할 수 있습니다. 하지만 지역 시장 상황, 경제 흐름, 법적 규제 등을 종합적으로 고려해야 합니다.
